# 04 - CAPTURE DES MESURES AU SOL

Dans ce notebook, nous allons capturer les mesures au sol uniquement et ajouter d'autres informations

In [1]:
# Chargement des librairies
import requests
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import tarfile
from notebookutils import mssparkutils # La gestion du lakehouse
import os
from pathlib import Path

StatementMeta(, 01915a5e-e869-42c6-80fd-62d41adaffd5, 3, Finished, Available, Finished, False)

## Lecture de la table silver nettoyé `clean_measures`

In [3]:
SILVER_TABLE_MEASURES = "abfss://0bcbec59-dbbb-41e1-a6a0-d3763e6c43e6@onelake.dfs.fabric.microsoft.com/23840c0a-00b7-4a0b-af79-896b6c9be935/Tables/silver/clean_measures"

StatementMeta(, 01915a5e-e869-42c6-80fd-62d41adaffd5, 5, Finished, Available, Finished, False)

In [4]:
# Lecture de la table brute à nettoyer
df_measures = spark.read.format("delta").load(SILVER_TABLE_MEASURES)

StatementMeta(, 01915a5e-e869-42c6-80fd-62d41adaffd5, 6, Finished, Available, Finished, False)

## Filtrage sur les mesures au sol

In [5]:
# DataFrame sur les mesures au sol
ground_measures = (df_measures
                        .filter("qualification = 'groundlevel'")
                        .drop("departureTime", "arrivalTime", "airportOrigin", 
                        "airportDestination", "aircraftType", "firstLatitude",
                        "firstLongitude", "midLatitude", "midLongitude",
                        "lastLatitude", "lastLongitude"))

StatementMeta(, 01915a5e-e869-42c6-80fd-62d41adaffd5, 7, Finished, Available, Finished, False)

In [7]:
ground_measures.printSchema()

StatementMeta(, 01915a5e-e869-42c6-80fd-62d41adaffd5, 9, Finished, Available, Finished, False)

root
 |-- reportUuid: string (nullable = true)
 |-- apparatusId: string (nullable = true)
 |-- apparatusVersion: string (nullable = true)
 |-- apparatusSensorType: string (nullable = true)
 |-- apparatusTubeType: string (nullable = true)
 |-- value: float (nullable = true)
 |-- hitsNumber: string (nullable = true)
 |-- calibrationFunction: string (nullable = true)
 |-- startTime: timestamp (nullable = true)
 |-- endTime: timestamp (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- altitude: double (nullable = true)
 |-- endLatitude: double (nullable = true)
 |-- endLongitude: double (nullable = true)
 |-- endAltitude: double (nullable = true)
 |-- endAccuracy: double (nullable = true)
 |-- manualReporting: double (nullable = true)
 |-- organisationReporting: string (nullable = true)
 |-- description: string (nullable = true)
 |-- userId: string (nullable = true)
 |-- measurementEnvironment: str

## Ecriture dans la table silver `ground_measures`

In [6]:
ground_measures.write.format("delta").mode("overwrite").saveAsTable("silver.ground_measures")

StatementMeta(, 01915a5e-e869-42c6-80fd-62d41adaffd5, 8, Finished, Available, Finished, False)